# Copy number on a whole transcriptome section with Atera

TEXT_INTRO

In [ ]:
import importlib.metadata
import os
import time

import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import scanpy as sc
from sklearn.metrics import adjusted_rand_score

import infercnvpy as cnv

sc.set_figure_params(dpi=50, frameon=False, figsize=(4.5, 4.5))

## Load the data

TEXT_LOAD

In [ ]:
adata = cnv.datasets.wu2026_atera_breast()
print(adata.uns["source"]["dataset"]["attribution"])
counts_per_cell = np.asarray(adata.X.sum(axis=1)).ravel()
print(f"median counts per cell: {np.median(counts_per_cell):.0f}")
adata

TEXT_TYPES

In [ ]:
cell_type = adata.obs["cell_type"].astype(str)
normal_states = ["Normal Luminal", "Normal Basal", "Ductal Luminal"]
normal_epithelium = (cell_type == "epithelial") & adata.obs["pred_state"].isin(normal_states)
adata.obs["cell_type_ref"] = pd.Categorical(cell_type.where(~normal_epithelium, "normal epithelium"))
# for the comparisons below: the other epithelial cells by the clone group of the WGS region around them
clone_group = adata.obs["wgs_clone_group"].astype(str)
other_epithelium = (cell_type == "epithelial") & ~normal_epithelium
adata.obs["group"] = adata.obs["cell_type_ref"].astype(str)
adata.obs.loc[other_epithelium, "group"] = np.where(
    clone_group[other_epithelium].isin(["A", "C", "N"]),
    "epithelium in " + clone_group[other_epithelium],
    "other epithelium",
)
pd.crosstab(adata.obs["cell_type_ref"], adata.obs["wgs_clone_group"].astype(str), margins=True)

In [ ]:
cnv.pl.spatial_cnv(adata, color=["cell_type_ref", "wgs_clone_group"], size=1, wspace=0.6)

TEXT_WGS

In [ ]:
wgs = adata.uns["wgs_truth"]["arms"].query("kind == 'arm'")
wgs = wgs.pivot(index="arm", columns="clone_group", values="log2")
arms = sorted(wgs.index, key=lambda arm: (int(arm[:-1]), arm[-1]))
wgs = wgs.loc[arms]
wgs.T.round(2)

## Copy number without and with a reference

TEXT_RUNS

In [ ]:
# GRCh38 centromere midpoints (Mb), to assign each gene to an arm
centromere_mb = {
    "chr1": 123.4, "chr2": 93.9, "chr3": 90.9, "chr4": 50.0, "chr5": 48.8, "chr6": 59.8, "chr7": 60.1, "chr8": 45.2,
    "chr9": 43.0, "chr10": 39.8, "chr11": 53.4, "chr12": 35.5, "chr13": 17.7, "chr14": 17.2, "chr15": 19.0,
    "chr16": 36.8, "chr17": 25.1, "chr18": 18.5, "chr19": 26.2, "chr20": 28.1, "chr21": 12.0, "chr22": 15.0,
}  # fmt: skip
chromosome = adata.var["chromosome"].astype(str)
side = np.where(adata.var["start"] / 1e6 < chromosome.map(centromere_mb), "p", "q")
gene_arm = pd.Series(chromosome.str.removeprefix("chr") + side, index=adata.var_names).where(
    chromosome.isin(centromere_mb)
)
epithelial = (cell_type == "epithelial").to_numpy()


def arm_profiles(key):
    """Mean gene value per arm of the epithelial cells of each clone group; removes the gene values layer."""
    gene_values = adata.layers.pop(f"gene_values_{key}")
    used = adata.uns[key]["genes"].index
    columns = adata.var_names.get_indexer(used)
    profiles = {}
    for group in ["A", "C", "N"]:
        cells = epithelial & (clone_group == group).to_numpy()
        gene_mean = pd.Series(np.nanmean(gene_values[np.ix_(cells, columns)], axis=0), index=used)
        profiles[group] = gene_mean.groupby(gene_arm[used]).mean()
    return pd.DataFrame(profiles).loc[arms]


def wgs_correlation(profile):
    """Pearson r over the arms between the pseudobulk profiles and the WGS, per clone group and for A minus C."""
    r = {f"r {group}": np.corrcoef(profile[group], wgs[group])[0, 1] for group in ["A", "C"]}
    r["r A-C"] = np.corrcoef(profile["A"] - profile["C"], wgs["A"] - wgs["C"])[0, 1]
    return r


immune_stromal = ["T/NK", "B/plasma", "myeloid", "endothelial", "fibroblast"]
all_normal = [*immune_stromal, "smooth muscle/pericyte", "normal epithelium"]
runs = {
    "no reference": {},
    "immune, endothelial, fibroblast": {"reference_key": "cell_type_ref", "reference_cat": immune_stromal},
    "all normal types": {"reference_key": "cell_type_ref", "reference_cat": all_normal},
}
keys = dict(zip(runs, ["cn_none", "cn_immune_stromal", "cn_all_normal"], strict=True))
profiles, correlations = {}, {}
for name, kwargs in runs.items():
    start = time.perf_counter()
    cnv.tl.copy_number(adata, calculate_gene_values=True, key_added=keys[name], show_progress=False, **kwargs)
    print(f"{name}: {time.perf_counter() - start:.0f} s")
    profiles[name] = arm_profiles(keys[name])
    correlations[name] = wgs_correlation(profiles[name])

In [ ]:
order = [
    "T/NK", "B/plasma", "myeloid", "endothelial", "fibroblast", "smooth muscle/pericyte", "unassigned",
    "normal epithelium", "epithelium in A", "epithelium in C", "epithelium in N", "other epithelium",
]  # fmt: skip
aneuploid = pd.DataFrame(
    {name: (adata.obs[key] == "aneuploid").groupby(adata.obs["group"]).mean() for name, key in keys.items()}
).loc[order]
aneuploid.insert(0, "cells", adata.obs["group"].value_counts())
aneuploid.round(3)

TEXT_CALLS

In [ ]:
pd.DataFrame(correlations).T.round(2)

TEXT_R

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, group in zip(axes, ["A", "C"], strict=True):
    profile = profiles["all normal types"][group]
    ax.axhline(0, color="lightgray", lw=0.8)
    ax.axvline(0, color="lightgray", lw=0.8)
    ax.scatter(wgs[group], profile, s=12)
    for arm in arms:
        if abs(wgs.loc[arm, group]) > 0.3:
            ax.annotate(arm, (wgs.loc[arm, group], profile[arm]), fontsize=8)
    ax.set(xlabel="WGS log2 ratio", ylabel="pseudobulk log ratio (natural log)", title=f"clone group {group}")
    ax.grid(False)
plt.tight_layout()

TEXT_HEATMAP

In [ ]:
cnv.pl.chromosome_heatmap(adata, groupby="cn_all_normal_clone", use_rep="cn_all_normal", figsize=(12, 3))
adata.uns["cn_all_normal"]["clones"].round(3)

In [ ]:
cnv.pl.spatial_cnv(adata, color=["cn_none", "cn_all_normal", "cn_all_normal_clone"], size=1, wspace=0.4)

## A normal type left out of the reference

TEXT_PITFALL

In [ ]:
pd.crosstab(adata.obs["group"], adata.obs["cn_immune_stromal_clone"]).loc[order]

TEXT_PITFALL2

## Restarts

TEXT_RESTARTS

In [ ]:
fits = {"n_restarts=5": adata.obs[["cn_none", "cn_none_clone"]].set_axis(["call", "clone"], axis=1)}
for seed in [0, 1, 2]:
    obs, _, _, _ = cnv.tl.copy_number(adata, n_restarts=1, random_state=seed, inplace=False, show_progress=False)
    fits[f"n_restarts=1, seed {seed}"] = obs[["copy_number", "copy_number_clone"]].set_axis(["call", "clone"], axis=1)
called = np.logical_and.reduce([fit["call"].isin(["aneuploid", "diploid"]).to_numpy() for fit in fits.values()])
summary = pd.DataFrame(
    {
        name: {
            "aneuploid cells": (fit["call"] == "aneuploid").sum(),
            "aneuploid clones": sum(clone.startswith("T") for clone in fit["clone"].astype(str).unique()),
        }
        for name, fit in fits.items()
    }
).T
same_call = pd.DataFrame(
    {a: {b: (fits[a]["call"] == fits[b]["call"])[called].mean() for b in fits} for a in fits}
)
clone_ari = pd.DataFrame(
    {a: {b: adjusted_rand_score(fits[a]["clone"][called], fits[b]["clone"][called]) for b in fits} for a in fits}
)
print(f"cells called in every fit: {called.sum()} of {adata.n_obs}")
summary

In [ ]:
print("share of cells with the same aneuploid or diploid call")
print(same_call.round(3).to_string())
print("\nadjusted Rand index of the clone labels")
print(clone_ari.round(2).to_string())

TEXT_RESTARTS2

In [ ]:
restarts = adata.uns["cn_none"]["restarts"]
print(f"fit kept: {restarts['chosen']}, aneuploid cells per fit: {restarts['n_aneuploid'].tolist()}")
print(pd.DataFrame(restarts["agreement"]).round(3).to_string())
stability = adata.obs["cn_none_stability"]
print(f"cells with a stability below 1: {(stability < 1).mean():.1%}")
print(stability[stability < 1].groupby(adata.obs["group"][stability < 1]).size().loc[lambda s: s > 0].to_string())
cnv.pl.spatial_cnv(adata, color="cn_none_stability", size=1, cmap="viridis", title="stability, no reference")

TEXT_RESTARTS3

## Detectability

TEXT_DETECT

In [ ]:
cnv.tl.detectability(adata, groupby="cell_type_ref", reference_key="cell_type_ref", reference_cat=all_normal)
table = adata.uns["detectability"]["table"]
table.groupby("group", observed=True)[["cells_gain", "cells_loss"]].median().round(1)

TEXT_DETECT2

In [ ]:
# versions and threads used to run this notebook; INFERCNVPY_COMMIT is set by the docs build
print("infercnvpy", cnv.__version__, os.environ.get("INFERCNVPY_COMMIT", ""))
for package in ["anndata", "scanpy", "numpy", "pandas", "scipy", "scikit-learn", "numba"]:
    print(package, importlib.metadata.version(package))
print("numba threads", numba.get_num_threads(), "| CPUs", os.cpu_count())